# Hierarchical Contract Clause Segmenter — CUAD v1
**LegalBERT → Transformer doc encoder → CRF boundary detector**

Complete self-contained pipeline. Run every cell top-to-bottom, or use `Runtime → Run all`.

| Section | Content |
|---------|---------|
| 1 | Install & imports |
| 2 | Sentence splitter |
| 3 | Dataset & tokenisation |
| 4 | Model |
| 5 | Evaluation metrics |
| 6 | CUAD loader |
| 7 | Data preparation |
| 8 | Training loop |
| 9 | End-to-end run ← **set CUAD_DIR here** |
| 10 | Inference demo |

> Before running Section 9, set `CUAD_DIR` to the path of your `CUAD_v1` folder.

## 1. Install & Imports

In [ ]:
import subprocess, sys
pkgs = ["torch", "transformers", "pytorch-crf", "tokenizers", "huggingface_hub"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=False)

In [ ]:
import csv, dataclasses, difflib, json, logging, random, re
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
from torchcrf import CRF
from transformers import AutoModel, AutoTokenizer, PreTrainedTokenizerBase

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)s  %(message)s")
log = logging.getLogger(__name__)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 2. Legal-Aware Sentence Splitter

In [ ]:
import re
from dataclasses import dataclass
from typing import List, Optional


@dataclass
class Sentence:
    text: str
    start_char: int
    end_char: int
    is_header: bool = False          # all-caps / bold-like formatting
    relative_position: float = 0.0   # 0.0 → 1.0 within document
    follows_header: bool = False
    indent_level: int = 0            # 0 = top level, 1 = sub-clause, etc.


# Patterns that signal a clause/section boundary (not mid-sentence)
_CLAUSE_NUMBER_PATTERNS = [
    r"^\s*\d+\.\s",                          # "1. "
    r"^\s*\d+\.\d+\s",                       # "1.1 "
    r"^\s*\d+\.\d+\.\d+\s",                  # "1.1.1 "
    r"^\s*[A-Z]\.\s",                        # "A. "
    r"^\s*\([a-z]\)\s",                      # "(a) "
    r"^\s*\([ivx]+\)\s",                     # "(iv) "
    r"^\s*Article\s+\d+",                    # "Article 12"
    r"^\s*Section\s+\d+",                    # "Section 3"
    r"^\s*Schedule\s+",                      # "Schedule A"
    r"^\s*Exhibit\s+",                       # "Exhibit B"
    r"^\s*Annex\s+",                         # "Annex 1"
    r"^\s*WHEREAS",                          # recitals
    r"^\s*NOW,?\s+THEREFORE",               # operative clause start
]

_COMPILED_CLAUSE_PATTERNS = [re.compile(p, re.IGNORECASE) for p in _CLAUSE_NUMBER_PATTERNS]

_HEADER_PATTERN = re.compile(r"^[A-Z\s\d\.\:]{5,80}$")   # all-caps line
_INDENT_PATTERN = re.compile(r"^(\s+)")


def _indent_level(line: str) -> int:
    m = _INDENT_PATTERN.match(line)
    if not m:
        return 0
    spaces = len(m.group(1).expandtabs(4))
    return spaces // 4   # bucket into levels of 4 spaces


def _is_clause_start(line: str) -> bool:
    stripped = line.strip()
    if not stripped:
        return False
    return any(p.match(stripped) for p in _COMPILED_CLAUSE_PATTERNS)


def _is_header(line: str) -> bool:
    stripped = line.strip()
    if not stripped:
        return False
    return bool(_HEADER_PATTERN.match(stripped)) and len(stripped) > 4


def _split_into_raw_lines(text: str) -> List[tuple]:
    """Return (line_text, char_offset) pairs."""
    lines = []
    offset = 0
    for line in text.splitlines(keepends=True):
        lines.append((line, offset))
        offset += len(line)
    return lines


def _merge_continuation_lines(raw_lines: List[tuple]) -> List[tuple]:
    """
    Merge lines that are clearly continuations (don't start a new clause/header,
    and the previous line doesn't end with sentence-terminal punctuation).
    Returns list of (merged_text, start_offset).
    """
    terminal_re = re.compile(r"[.!?;]\s*$")
    merged = []
    buf_text = ""
    buf_start = 0

    for line_text, offset in raw_lines:
        stripped = line_text.strip()
        if not stripped:
            # blank line → flush buffer as paragraph boundary
            if buf_text.strip():
                merged.append((buf_text, buf_start))
            buf_text = ""
            buf_start = offset
            continue

        if not buf_text:
            buf_text = line_text
            buf_start = offset
        elif _is_clause_start(stripped) or _is_header(stripped):
            if buf_text.strip():
                merged.append((buf_text, buf_start))
            buf_text = line_text
            buf_start = offset
        else:
            buf_text = buf_text.rstrip("\n") + " " + stripped + "\n"

    if buf_text.strip():
        merged.append((buf_text, buf_start))

    return merged


def split_contract(text: str) -> List[Sentence]:
    """
    Main entry point.  Returns a list of Sentence objects with metadata.
    """
    raw_lines = _split_into_raw_lines(text)
    merged = _merge_continuation_lines(raw_lines)

    sentences: List[Sentence] = []
    doc_len = max(len(text), 1)

    prev_is_header = False
    for seg_text, start in merged:
        stripped = seg_text.strip()
        if not stripped:
            prev_is_header = False
            continue

        end = start + len(seg_text)
        is_hdr = _is_header(stripped)
        indent = _indent_level(seg_text)

        s = Sentence(
            text=stripped,
            start_char=start,
            end_char=end,
            is_header=is_hdr,
            relative_position=start / doc_len,
            follows_header=prev_is_header,
            indent_level=indent,
        )
        sentences.append(s)
        prev_is_header = is_hdr

    return sentences

## 3. Dataset & Tokenisation

In [ ]:
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch
from torch.utils.data import Dataset
from transformers import PreTrainedTokenizerBase



# ── label maps ──────────────────────────────────────────────────────────────

BIO_LABEL2ID = {"O": 0, "B": 1}
BIO_ID2LABEL = {v: k for k, v in BIO_LABEL2ID.items()}
NUM_LABELS = len(BIO_LABEL2ID)   # 2 for binary boundary detection


# ── raw data schema ──────────────────────────────────────────────────────────

@dataclass
class RawContract:
    """
    A single annotated contract.

    `clauses` is a list of (start_char, end_char) tuples marking the span of
    each clause in `text`.  The very first character of each clause span
    determines which sentence gets label B=1.
    """
    doc_id: str
    text: str
    clauses: List[Tuple[int, int]]   # [(start, end), ...]


# ── feature extraction ───────────────────────────────────────────────────────

def _build_positional_features(sentences: List[Sentence]) -> torch.Tensor:
    """
    Returns a float tensor of shape [N, 4]:
      [relative_position, is_header, follows_header, indent_level_norm]
    """
    feats = []
    for s in sentences:
        feats.append([
            s.relative_position,
            float(s.is_header),
            float(s.follows_header),
            min(s.indent_level / 4.0, 1.0),   # normalise to [0, 1]
        ])
    return torch.tensor(feats, dtype=torch.float)


def _assign_bio_labels(
    sentences: List[Sentence],
    clauses: List[Tuple[int, int]],
) -> List[int]:
    """
    For each sentence, assign B=1 if it contains the start of a clause.
    Otherwise O=0.
    """
    clause_starts = {start for start, _ in clauses}
    labels = []
    for s in sentences:
        # sentence is a clause start if any clause boundary falls within it
        label = int(
            any(clause_starts[c] >= s.start_char and clause_starts[c] < s.end_char
                for c in clause_starts
                if s.start_char <= clause_starts[c] < s.end_char)
            if clause_starts else 0
        )
        # simpler: check overlap
        hit = 0
        for cs in clause_starts:
            if s.start_char <= cs < s.end_char:
                hit = 1
                break
        labels.append(hit)
    return labels


# ── tokenised document ───────────────────────────────────────────────────────

@dataclass
class TokenisedDocument:
    doc_id: str
    input_ids: List[torch.Tensor]          # [N] tensors of shape [seq_len]
    attention_masks: List[torch.Tensor]    # [N] tensors of shape [seq_len]
    positional_features: torch.Tensor      # [N, 4]
    labels: torch.Tensor                   # [N]  long
    num_sentences: int


def tokenise_contract(
    raw: RawContract,
    tokenizer: PreTrainedTokenizerBase,
    max_sent_len: int = 128,
) -> TokenisedDocument:
    sentences = split_contract(raw.text)
    labels = _assign_bio_labels(sentences, raw.clauses)
    pos_feats = _build_positional_features(sentences)

    input_ids_list = []
    attn_mask_list = []

    for sent in sentences:
        enc = tokenizer(
            sent.text,
            max_length=max_sent_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        input_ids_list.append(enc["input_ids"].squeeze(0))      # [seq_len]
        attn_mask_list.append(enc["attention_mask"].squeeze(0))

    return TokenisedDocument(
        doc_id=raw.doc_id,
        input_ids=input_ids_list,
        attention_masks=attn_mask_list,
        positional_features=pos_feats,
        labels=torch.tensor(labels, dtype=torch.long),
        num_sentences=len(sentences),
    )


# ── PyTorch Dataset ───────────────────────────────────────────────────────────

class ContractDataset(Dataset):
    """
    Each item is one full contract (variable-length sentence sequence).
    Collation handles padding to the longest document in each batch.
    """

    def __init__(
        self,
        documents: List[TokenisedDocument],
        max_doc_sentences: int = 512,
    ):
        self.documents = documents
        self.max_doc_sentences = max_doc_sentences

    def __len__(self) -> int:
        return len(self.documents)

    def __getitem__(self, idx: int) -> TokenisedDocument:
        return self.documents[idx]

    # ── factory methods ──────────────────────────────────────────────────────

    @classmethod
    def from_jsonl(
        cls,
        path: str | Path,
        tokenizer: PreTrainedTokenizerBase,
        max_sent_len: int = 128,
        max_doc_sentences: int = 512,
    ) -> "ContractDataset":
        """
        Load from a JSONL file.  Each line must be a JSON object:
        {
          "doc_id": "contract_001",
          "text": "THIS AGREEMENT ...",
          "clauses": [[0, 512], [513, 1024], ...]
        }
        """
        docs = []
        with open(path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                obj = json.loads(line)
                raw = RawContract(
                    doc_id=obj["doc_id"],
                    text=obj["text"],
                    clauses=[tuple(c) for c in obj["clauses"]],
                )
                tok = tokenise_contract(raw, tokenizer, max_sent_len)
                # truncate very long documents
                if tok.num_sentences > max_doc_sentences:
                    tok = _truncate_document(tok, max_doc_sentences)
                docs.append(tok)
        return cls(docs, max_doc_sentences)


def _truncate_document(doc: TokenisedDocument, max_n: int) -> TokenisedDocument:
    return TokenisedDocument(
        doc_id=doc.doc_id,
        input_ids=doc.input_ids[:max_n],
        attention_masks=doc.attention_masks[:max_n],
        positional_features=doc.positional_features[:max_n],
        labels=doc.labels[:max_n],
        num_sentences=min(doc.num_sentences, max_n),
    )


# ── Collate function ─────────────────────────────────────────────────────────

def collate_contracts(
    batch: List[TokenisedDocument],
) -> Dict[str, torch.Tensor]:
    """
    Pads each document to the longest document in the batch along the sentence
    dimension.  Returns:
      input_ids       : [B, max_N, seq_len]
      attention_mask  : [B, max_N, seq_len]  (sentence tokenizer mask)
      sentence_mask   : [B, max_N]           (1 = real sentence, 0 = padding)
      pos_features    : [B, max_N, 4]
      labels          : [B, max_N]  (-100 for padded positions)
    """
    max_n = max(d.num_sentences for d in batch)
    seq_len = batch[0].input_ids[0].shape[0]   # all padded to same sent len

    B = len(batch)
    input_ids = torch.zeros(B, max_n, seq_len, dtype=torch.long)
    attn_mask = torch.zeros(B, max_n, seq_len, dtype=torch.long)
    sent_mask  = torch.zeros(B, max_n, dtype=torch.bool)
    pos_feats  = torch.zeros(B, max_n, 4, dtype=torch.float)
    labels     = torch.full((B, max_n), fill_value=-100, dtype=torch.long)

    for i, doc in enumerate(batch):
        n = doc.num_sentences
        input_ids[i, :n] = torch.stack(doc.input_ids)
        attn_mask[i, :n] = torch.stack(doc.attention_masks)
        sent_mask[i, :n]  = True
        pos_feats[i, :n]  = doc.positional_features
        labels[i, :n]     = doc.labels

    return {
        "input_ids":      input_ids,
        "attention_mask": attn_mask,
        "sentence_mask":  sent_mask,
        "pos_features":   pos_feats,
        "labels":         labels,
    }

## 4. Model (LegalBERT + Doc Encoder + CRF)

In [ ]:
from dataclasses import dataclass
from typing import Optional, Tuple

import torch
import torch.nn as nn
from transformers import AutoModel, PreTrainedModel
from torchcrf import CRF   # pip install pytorch-crf


# ── Config ────────────────────────────────────────────────────────────────────

@dataclass
class SegmenterConfig:
    # Sentence encoder
    encoder_model_name: str = "nlpaueb/legal-bert-base-uncased"
    freeze_encoder_epochs: int = 2          # freeze LegalBERT for first N epochs

    # Positional features
    pos_feature_dim: int = 4                # must match dataset._build_positional_features

    # Projection (sentence emb + pos features → doc encoder input)
    proj_hidden_size: int = 256

    # Document-level encoder
    doc_encoder_type: str = "transformer"   # "transformer" | "bilstm"
    doc_encoder_layers: int = 2
    doc_encoder_heads: int = 4             # transformer only
    doc_encoder_ff_dim: int = 512          # transformer only
    doc_encoder_dropout: float = 0.1

    # Classification head
    num_labels: int = 2                    # B / O
    head_dropout: float = 0.2

    # CRF
    use_crf: bool = True

    # Focal loss
    use_focal_loss: bool = False
    focal_gamma: float = 2.0
    boundary_pos_weight: float = 8.0       # BCE pos_weight (imbalance correction)


# ── Focal Loss ────────────────────────────────────────────────────────────────

class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, weight: Optional[torch.Tensor] = None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        logits  : [N, C]
        targets : [N]  (ignore_index = -100)
        """
        mask = targets != -100
        logits  = logits[mask]
        targets = targets[mask]

        log_p = torch.log_softmax(logits, dim=-1)
        p     = torch.exp(log_p)
        focal_weight = (1 - p) ** self.gamma
        loss = nn.NLLLoss(weight=self.weight, reduction="mean")(
            (focal_weight * log_p), targets
        )
        return loss


# ── Document-level encoders ───────────────────────────────────────────────────

class TransformerDocEncoder(nn.Module):
    def __init__(self, cfg: SegmenterConfig):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.proj_hidden_size,
            nhead=cfg.doc_encoder_heads,
            dim_feedforward=cfg.doc_encoder_ff_dim,
            dropout=cfg.doc_encoder_dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=cfg.doc_encoder_layers
        )

    def forward(
        self,
        x: torch.Tensor,              # [B, N, D]
        src_key_padding_mask: torch.Tensor,  # [B, N]  True = IGNORE
    ) -> torch.Tensor:                # [B, N, D]
        return self.encoder(x, src_key_padding_mask=src_key_padding_mask)


class BiLSTMDocEncoder(nn.Module):
    def __init__(self, cfg: SegmenterConfig):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=cfg.proj_hidden_size,
            hidden_size=cfg.proj_hidden_size // 2,
            num_layers=cfg.doc_encoder_layers,
            batch_first=True,
            bidirectional=True,
            dropout=cfg.doc_encoder_dropout if cfg.doc_encoder_layers > 1 else 0.0,
        )

    def forward(
        self,
        x: torch.Tensor,
        src_key_padding_mask: torch.Tensor,
    ) -> torch.Tensor:
        # pack for efficiency
        lengths = (~src_key_padding_mask).sum(dim=1).cpu()
        packed  = nn.utils.rnn.pack_padded_sequence(
            x, lengths, batch_first=True, enforce_sorted=False
        )
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        # re-pad to original max length
        B, N, D = x.shape
        if out.shape[1] < N:
            pad = torch.zeros(B, N - out.shape[1], D, device=x.device)
            out = torch.cat([out, pad], dim=1)
        return out


# ── Main model ────────────────────────────────────────────────────────────────

class HierarchicalClauseSegmenter(nn.Module):
    def __init__(self, cfg: SegmenterConfig):
        super().__init__()
        self.cfg = cfg

        # 1. Sentence encoder
        self.sentence_encoder: PreTrainedModel = AutoModel.from_pretrained(
            cfg.encoder_model_name
        )
        bert_hidden = self.sentence_encoder.config.hidden_size

        # 2. Positional feature projection (fuse with CLS)
        self.feature_proj = nn.Sequential(
            nn.Linear(bert_hidden + cfg.pos_feature_dim, cfg.proj_hidden_size),
            nn.LayerNorm(cfg.proj_hidden_size),
            nn.GELU(),
        )

        # 3. Document encoder
        if cfg.doc_encoder_type == "transformer":
            self.doc_encoder = TransformerDocEncoder(cfg)
        elif cfg.doc_encoder_type == "bilstm":
            self.doc_encoder = BiLSTMDocEncoder(cfg)
        else:
            raise ValueError(f"Unknown doc_encoder_type: {cfg.doc_encoder_type}")

        # 4. Classification head
        self.head = nn.Sequential(
            nn.Dropout(cfg.head_dropout),
            nn.Linear(cfg.proj_hidden_size, cfg.num_labels),
        )

        # 5. Optional CRF
        self.crf = CRF(cfg.num_labels, batch_first=True) if cfg.use_crf else None

        # 6. Loss
        if cfg.use_focal_loss:
            self.loss_fn = FocalLoss(gamma=cfg.focal_gamma)
        else:
            weight = torch.tensor([1.0, cfg.boundary_pos_weight])
            self.loss_fn = nn.CrossEntropyLoss(weight=weight, ignore_index=-100)

    # ── Sentence encoding ─────────────────────────────────────────────────────

    def _encode_sentences(
        self,
        input_ids: torch.Tensor,      # [B, N, seq_len]
        attention_mask: torch.Tensor, # [B, N, seq_len]
    ) -> torch.Tensor:                # [B, N, bert_hidden]
        B, N, L = input_ids.shape
        # flatten to [B*N, L]
        flat_ids  = input_ids.view(B * N, L)
        flat_mask = attention_mask.view(B * N, L)

        out = self.sentence_encoder(
            input_ids=flat_ids,
            attention_mask=flat_mask,
        )
        cls = out.last_hidden_state[:, 0, :]   # [B*N, H]
        return cls.view(B, N, -1)              # [B, N, H]

    # ── Forward ───────────────────────────────────────────────────────────────

    def forward(
        self,
        input_ids: torch.Tensor,         # [B, N, seq_len]
        attention_mask: torch.Tensor,    # [B, N, seq_len]
        sentence_mask: torch.Tensor,     # [B, N]  True = real sentence
        pos_features: torch.Tensor,      # [B, N, 4]
        labels: Optional[torch.Tensor] = None,   # [B, N]
    ) -> Tuple[Optional[torch.Tensor], torch.Tensor]:
        """
        Returns (loss, logits).
          loss   : scalar or None (if labels not provided)
          logits : [B, N, num_labels]
        """
        # 1. Sentence embeddings
        cls_emb = self._encode_sentences(input_ids, attention_mask)  # [B, N, H]

        # 2. Fuse with positional features
        fused = torch.cat([cls_emb, pos_features], dim=-1)            # [B, N, H+4]
        proj  = self.feature_proj(fused)                              # [B, N, D]

        # 3. Document-level context
        #    TransformerEncoder uses True = *ignore*, so invert sentence_mask
        padding_mask = ~sentence_mask   # [B, N]
        ctx = self.doc_encoder(proj, src_key_padding_mask=padding_mask)  # [B, N, D]

        # 4. Per-sentence logits
        logits = self.head(ctx)   # [B, N, num_labels]

        # 5. Loss
        loss = None
        if labels is not None:
            if self.crf is not None:
                # CRF expects mask as ByteTensor
                crf_mask = sentence_mask.bool()
                # replace -100 labels with 0 for CRF (they're masked out)
                safe_labels = labels.clone()
                safe_labels[safe_labels == -100] = 0
                loss = -self.crf(logits, safe_labels, mask=crf_mask, reduction="mean")
            else:
                B, N, C = logits.shape
                loss = self.loss_fn(logits.view(B * N, C), labels.view(B * N))

        return loss, logits

    # ── Inference ─────────────────────────────────────────────────────────────

    @torch.no_grad()
    def predict(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        sentence_mask: torch.Tensor,
        pos_features: torch.Tensor,
        threshold: float = 0.5,
    ) -> torch.Tensor:
        """
        Returns predicted label tensor [B, N] using threshold (non-CRF) or
        Viterbi decode (CRF).
        """
        _, logits = self.forward(
            input_ids, attention_mask, sentence_mask, pos_features
        )

        if self.crf is not None:
            preds = self.crf.decode(logits, mask=sentence_mask.bool())
            # pad to max_N
            B, N = logits.shape[:2]
            out = torch.zeros(B, N, dtype=torch.long)
            for i, seq in enumerate(preds):
                out[i, :len(seq)] = torch.tensor(seq)
            return out
        else:
            probs = torch.softmax(logits, dim=-1)[:, :, 1]  # P(B) per sentence
            return (probs >= threshold).long()

    # ── Encoder freeze / unfreeze ─────────────────────────────────────────────

    def freeze_encoder(self):
        for p in self.sentence_encoder.parameters():
            p.requires_grad = False

    def unfreeze_encoder(self):
        for p in self.sentence_encoder.parameters():
            p.requires_grad = True

## 5. Evaluation Metrics

In [ ]:
from typing import Dict, List


# ── Boundary P / R / F1 ───────────────────────────────────────────────────────

def _boundary_prf(
    preds: List[int], labels: List[int]
) -> Dict[str, float]:
    tp = fp = fn = 0
    for p, g in zip(preds, labels):
        if p == 1 and g == 1:
            tp += 1
        elif p == 1 and g == 0:
            fp += 1
        elif p == 0 and g == 1:
            fn += 1
    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    return {"precision": precision, "recall": recall, "boundary_f1": f1}


# ── WindowDiff ────────────────────────────────────────────────────────────────

def _window_diff(ref: List[int], hyp: List[int], k: Optional[int] = None) -> float:
    """
    WindowDiff measures near-miss segmentation errors.
    Lower is better (0 = perfect).

    k defaults to half the average true segment length.
    """
    assert len(ref) == len(hyp), "sequences must be same length"
    N = len(ref)
    if N < 2:
        return 0.0

    if k is None:
        num_boundaries = max(sum(ref), 1)
        k = max(1, N // (2 * num_boundaries))

    errors = 0
    total  = 0
    for i in range(N - k):
        ref_count = sum(ref[i + 1: i + k + 1])
        hyp_count = sum(hyp[i + 1: i + k + 1])
        errors += int(ref_count != hyp_count)
        total  += 1

    return errors / max(total, 1)


# ── Pk ────────────────────────────────────────────────────────────────────────

def _pk(ref: List[int], hyp: List[int], k: Optional[int] = None) -> float:
    """
    Pk metric (Beeferman et al.).
    Lower is better (0 = perfect).
    """
    assert len(ref) == len(hyp)
    N = len(ref)
    if N < 2:
        return 0.0

    if k is None:
        num_boundaries = max(sum(ref), 1)
        k = max(1, N // (2 * num_boundaries))

    def same_segment(seq: List[int], i: int, j: int) -> bool:
        """True if positions i and j are in the same segment (no boundary between)."""
        return sum(seq[i + 1: j + 1]) == 0

    errors = 0
    total  = 0
    for i in range(N - k):
        j = i + k
        ref_same = same_segment(ref, i, j)
        hyp_same = same_segment(hyp, i, j)
        errors += int(ref_same != hyp_same)
        total  += 1

    return errors / max(total, 1)


# ── Corpus-level aggregation ──────────────────────────────────────────────────

def compute_metrics(
    all_preds:  List[List[int]],
    all_labels: List[List[int]],
) -> Dict[str, float]:
    """
    Compute metrics over a list of document-level predictions and labels.

    Args:
        all_preds  : list of per-document predicted label sequences
        all_labels : list of per-document ground-truth label sequences
                     (-100 padding is stripped automatically)

    Returns dict with keys:
        precision, recall, boundary_f1, window_diff, pk
    """
    flat_preds:  List[int] = []
    flat_labels: List[int] = []

    wd_scores: List[float] = []
    pk_scores: List[float] = []

    for preds, labels in zip(all_preds, all_labels):
        # strip padding (-100)
        valid = [(p, l) for p, l in zip(preds, labels) if l != -100]
        if not valid:
            continue
        p_seq, l_seq = zip(*valid)
        p_seq = list(p_seq)
        l_seq = list(l_seq)

        flat_preds.extend(p_seq)
        flat_labels.extend(l_seq)

        if len(l_seq) > 1:
            wd_scores.append(_window_diff(l_seq, p_seq))
            pk_scores.append(_pk(l_seq, p_seq))

    metrics = _boundary_prf(flat_preds, flat_labels)
    metrics["window_diff"] = sum(wd_scores) / max(len(wd_scores), 1)
    metrics["pk"]          = sum(pk_scores) / max(len(pk_scores), 1)
    return metrics


# ── Threshold sweep ───────────────────────────────────────────────────────────

def find_best_threshold(
    all_probs:  List[List[float]],   # P(boundary) per sentence, per doc
    all_labels: List[List[int]],
    metric: str = "boundary_f1",
    steps: int = 20,
) -> Tuple[float, Dict[str, float]]:
    """
    Sweep threshold ∈ [0.1, 0.9] and return the one maximising `metric`.

    Args:
        all_probs  : per-document lists of boundary probabilities
        all_labels : per-document ground-truth labels
        metric     : metric key to optimise (default: boundary_f1)
        steps      : number of threshold values to try

    Returns:
        (best_threshold, metrics_at_best_threshold)
    """
    best_t       = 0.5
    best_val     = -1.0
    best_metrics = {}

    for i in range(steps + 1):
        t = i / steps
        all_preds = [[int(p >= t) for p in probs] for probs in all_probs]
        m = compute_metrics(all_preds, all_labels)
        if m[metric] > best_val:
            best_val     = m[metric]
            best_t       = t
            best_metrics = m

    return best_t, best_metrics


# -- make Optional available (forgot import above)
from typing import Optional, Tuple

## 6. CUAD Data Loader

In [ ]:
import csv
import difflib
import logging
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

log = logging.getLogger(__name__)

# ── The 41 CUAD category column names (context columns in master_clauses.csv) ─

CUAD_CATEGORIES = [
    "Document Name",
    "Parties",
    "Agreement Date",
    "Effective Date",
    "Expiration Date",
    "Renewal Term",
    "Notice Period To Terminate Renewal",
    "Governing Law",
    "Most Favored Nation",
    "Non-Compete",
    "Exclusivity",
    "No-Solicit Of Customers",
    "Competitive Restriction Exception",
    "No-Solicit Of Employees",
    "Non-Disparagement",
    "Termination For Convenience",
    "Rofr/Rofo/Rofn",
    "Change Of Control",
    "Anti-Assignment",
    "Revenue/Profit Sharing",
    "Price Restrictions",
    "Minimum Commitment",
    "Volume Restriction",
    "Ip Ownership Assignment",
    "Joint Ip Ownership",
    "License Grant",
    "Non-Transferable License",
    "Affiliate License-Licensor",
    "Affiliate License-Licensee",
    "Unlimited/All-You-Can-Eat License",
    "Irrevocable Or Perpetual License",
    "Source Code Escrow",
    "Post-Termination Services",
    "Audit Rights",
    "Uncapped Liability",
    "Cap On Liability",
    "Liquidated Damages",
    "Warranty Duration",
    "Insurance",
    "Covenant Not To Sue",
    "Third Party Beneficiary",
]

# Columns in master_clauses.csv follow this pattern:
#   "<Category>" → context (clause text excerpt)
#   "<Category>-Answer" → human label (Yes/No/date/etc.)
# We only need the context columns.


@dataclass
class RawContract:
    """One CUAD contract with resolved clause character spans."""
    doc_id: str
    text: str
    clauses: List[Tuple[int, int]]            # sorted, non-overlapping (start, end)
    clause_types: Dict[int, List[str]] = field(default_factory=dict)
    # clause_types maps span_start → list of CUAD category names found there


# ── Fuzzy span finder ─────────────────────────────────────────────────────────

def _normalise(s: str) -> str:
    """Collapse whitespace for robust matching."""
    return re.sub(r"\s+", " ", s).strip()


def _find_span(full_text: str, excerpt: str, min_ratio: float = 0.85) -> Optional[Tuple[int, int]]:
    """
    Find the character span of `excerpt` inside `full_text`.

    1. Try exact substring match first (fast).
    2. Fall back to SequenceMatcher on normalised text (handles OCR noise).

    Returns (start, end) or None if not found.
    """
    excerpt = excerpt.strip()
    if not excerpt:
        return None

    # 1. Exact match
    idx = full_text.find(excerpt)
    if idx != -1:
        return (idx, idx + len(excerpt))

    # 2. Normalised exact match
    norm_full    = _normalise(full_text)
    norm_excerpt = _normalise(excerpt)
    idx = norm_full.find(norm_excerpt)
    if idx != -1:
        # Map back to original char offset (approximate — off by whitespace)
        return _remap_offset(full_text, norm_full, idx, idx + len(norm_excerpt))

    # 3. Fuzzy match on sliding windows of similar length
    n = len(norm_excerpt)
    step = max(1, n // 4)
    best_ratio = 0.0
    best_span  = None

    for start in range(0, len(norm_full) - n + 1, step):
        window = norm_full[start : start + n + n // 5]  # allow 20% length slop
        ratio  = difflib.SequenceMatcher(None, norm_excerpt, window, autojunk=False).ratio()
        if ratio > best_ratio:
            best_ratio = ratio
            best_span  = (start, start + len(window))

    if best_ratio >= min_ratio and best_span is not None:
        return _remap_offset(full_text, norm_full, best_span[0], best_span[1])

    return None


def _remap_offset(
    original: str, normalised: str, norm_start: int, norm_end: int
) -> Tuple[int, int]:
    """
    Map character offsets in the normalised string back to the original string.
    Uses a simple character-level scan.
    """
    orig_pos  = 0
    norm_pos  = 0
    start_map = {}
    end_map   = {}

    while orig_pos < len(original) and norm_pos < len(normalised):
        start_map[norm_pos] = orig_pos
        orig_ch = original[orig_pos]
        norm_ch = normalised[norm_pos]
        if orig_ch == norm_ch:
            orig_pos += 1
            norm_pos += 1
        elif orig_ch in (" ", "\n", "\t", "\r") and norm_ch == " ":
            # skip extra whitespace in original
            orig_pos += 1
        else:
            orig_pos += 1
            norm_pos += 1
        end_map[norm_pos] = orig_pos

    os = start_map.get(norm_start, norm_start)
    oe = end_map.get(norm_end, norm_end)
    return (min(os, len(original)), min(oe, len(original)))


# ── Span merging ──────────────────────────────────────────────────────────────

def _merge_overlapping(spans: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
    """Sort and merge overlapping/touching spans."""
    if not spans:
        return []
    spans = sorted(spans)
    merged = [spans[0]]
    for start, end in spans[1:]:
        if start <= merged[-1][1] + 1:   # overlapping or adjacent
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    return merged


# ── CSV parsing ───────────────────────────────────────────────────────────────

def _load_csv(csv_path: Path) -> List[Dict[str, str]]:
    """Load master_clauses.csv.  Returns list of row dicts."""
    rows = []
    with open(csv_path, encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)
    return rows


def _get_context_columns(fieldnames: List[str]) -> Dict[str, str]:
    """
    Map normalised category name → actual CSV column header.
    master_clauses.csv column names are slightly inconsistent across versions,
    so we do a case-insensitive prefix match.
    """
    mapping: Dict[str, str] = {}
    norm_fields = {f.lower().strip(): f for f in fieldnames}
    for cat in CUAD_CATEGORIES:
        key = cat.lower()
        # direct match
        if key in norm_fields:
            mapping[cat] = norm_fields[key]
            continue
        # partial match (column is a prefix of category or vice-versa)
        for nf, orig in norm_fields.items():
            # skip answer columns
            if nf.endswith("-answer") or nf.endswith(" answer"):
                continue
            if key.startswith(nf) or nf.startswith(key):
                mapping[cat] = orig
                break
    return mapping


# ── Main loader ───────────────────────────────────────────────────────────────

def load_cuad(
    cuad_root: str | Path,
    min_fuzzy_ratio: float = 0.85,
    max_contracts: Optional[int] = None,
    skip_empty: bool = True,
) -> List[RawContract]:
    """
    Load CUAD v1 dataset from `cuad_root` (the folder containing
    master_clauses.csv and full_contract_txt/).

    Args:
        cuad_root        : path to the CUAD_v1 directory
        min_fuzzy_ratio  : minimum similarity score for fuzzy span matching
        max_contracts    : limit number of contracts (useful for dev/testing)
        skip_empty       : skip contracts where no clause spans were found

    Returns:
        List of RawContract objects, ready for dataset.tokenise_contract().
    """
    cuad_root = Path(cuad_root)
    csv_path  = cuad_root / "master_clauses.csv"
    txt_dir   = cuad_root / "full_contract_txt"

    if not csv_path.exists():
        raise FileNotFoundError(f"master_clauses.csv not found at {csv_path}")
    if not txt_dir.exists():
        raise FileNotFoundError(f"full_contract_txt/ not found at {txt_dir}")

    log.info(f"Loading master_clauses.csv from {csv_path} …")
    rows = _load_csv(csv_path)
    if not rows:
        raise ValueError("master_clauses.csv is empty")

    col_map = _get_context_columns(list(rows[0].keys()))
    log.info(f"Matched {len(col_map)}/41 category columns in CSV")

    contracts: List[RawContract] = []
    total = min(len(rows), max_contracts) if max_contracts else len(rows)

    for i, row in enumerate(rows[:total]):
        # ── Identify the contract filename ────────────────────────────────────
        # Column 0 is the filename; it may have .pdf extension in some versions
        filename_col = list(row.keys())[0]
        raw_fname    = row[filename_col].strip()
        txt_fname    = Path(raw_fname).stem + ".txt"
        txt_path     = txt_dir / txt_fname

        if not txt_path.exists():
            log.warning(f"[{i+1}/{total}] TXT not found: {txt_path.name} — skipping")
            continue

        with open(txt_path, encoding="utf-8", errors="replace") as f:
            text = f.read()

        doc_id = Path(raw_fname).stem

        # ── Find all clause spans ─────────────────────────────────────────────
        all_spans: List[Tuple[int, int]] = []
        type_map:  Dict[int, List[str]]  = {}

        for cat, col in col_map.items():
            excerpt = row.get(col, "").strip()
            # CUAD stores multiple excerpts separated by newlines in some cells
            sub_excerpts = [e.strip() for e in excerpt.split("\n") if e.strip()]

            for sub in sub_excerpts:
                span = _find_span(text, sub, min_ratio=min_fuzzy_ratio)
                if span:
                    all_spans.append(span)
                    type_map.setdefault(span[0], []).append(cat)

        merged = _merge_overlapping(all_spans)

        if skip_empty and not merged:
            log.debug(f"[{i+1}/{total}] {doc_id}: no spans found — skipping")
            continue

        contracts.append(RawContract(
            doc_id=doc_id,
            text=text,
            clauses=merged,
            clause_types=type_map,
        ))

        if (i + 1) % 50 == 0 or (i + 1) == total:
            log.info(f"  Processed {i+1}/{total}  ({len(contracts)} kept)")

    log.info(f"Loaded {len(contracts)} contracts with clause annotations")
    return contracts

## 7. Data Preparation

In [ ]:
import json
import logging
import random
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List, Tuple


logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)s  %(message)s")
log = logging.getLogger(__name__)


# ── Contract type extraction from filename ────────────────────────────────────

def _contract_type(doc_id: str) -> str:
    """
    CUAD filenames end with the contract type, e.g.:
      AMAZONCOM_..._ServiceAgreement  →  ServiceAgreement
    """
    parts = doc_id.replace("-", "_").split("_")
    # last meaningful token(s)
    return parts[-1] if parts else "Unknown"


# ── Stratified split ──────────────────────────────────────────────────────────

def stratified_split(
    contracts: List[RawContract],
    val_frac:  float = 0.10,
    test_frac: float = 0.10,
    seed:      int   = 42,
) -> Tuple[List[RawContract], List[RawContract], List[RawContract]]:
    """
    Split contracts into train / val / test sets, stratified by contract type
    so each split has a representative mix.
    """
    rng = random.Random(seed)

    # Group by contract type
    by_type: Dict[str, List[RawContract]] = defaultdict(list)
    for c in contracts:
        ctype = _contract_type(c.doc_id)
        by_type[ctype].append(c)

    train, val, test = [], [], []

    for ctype, group in by_type.items():
        rng.shuffle(group)
        n     = len(group)
        n_val  = max(1, round(n * val_frac))  if n >= 3 else 0
        n_test = max(1, round(n * test_frac)) if n >= 3 else 0
        n_val  = min(n_val,  n // 3)
        n_test = min(n_test, n // 3)

        test  += group[:n_test]
        val   += group[n_test : n_test + n_val]
        train += group[n_test + n_val :]

    rng.shuffle(train)
    return train, val, test


# ── Serialisation ─────────────────────────────────────────────────────────────

def _contract_to_dict(c: RawContract) -> dict:
    return {
        "doc_id":  c.doc_id,
        "text":    c.text,
        "clauses": [list(span) for span in c.clauses],
    }


def write_jsonl(contracts: List[RawContract], path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for c in contracts:
            f.write(json.dumps(_contract_to_dict(c), ensure_ascii=False) + "\n")
    log.info(f"  Wrote {len(contracts):4d} contracts → {path}")


# ── Dataset statistics ────────────────────────────────────────────────────────

def compute_stats(
    train: List[RawContract],
    val:   List[RawContract],
    test:  List[RawContract],
) -> dict:
    def split_stats(contracts: List[RawContract]) -> dict:
        if not contracts:
            return {}
        clause_counts = [len(c.clauses) for c in contracts]
        text_lengths  = [len(c.text)    for c in contracts]
        # approximate sentence count (split on periods/newlines)
        import re
        sent_counts = [
            len(re.split(r"(?<=[.!?])\s+|\n{2,}", c.text))
            for c in contracts
        ]
        # boundary density = boundaries / sentences
        densities = [
            cc / max(sc, 1)
            for cc, sc in zip(clause_counts, sent_counts)
        ]

        return {
            "n_contracts":          len(contracts),
            "clauses_per_doc":      {
                "mean": round(sum(clause_counts) / len(clause_counts), 1),
                "min":  min(clause_counts),
                "max":  max(clause_counts),
            },
            "text_chars_per_doc":   {
                "mean": round(sum(text_lengths) / len(text_lengths)),
                "min":  min(text_lengths),
                "max":  max(text_lengths),
            },
            "sentences_per_doc":    {
                "mean": round(sum(sent_counts) / len(sent_counts), 1),
            },
            "boundary_density_mean": round(sum(densities) / len(densities), 4),
            "contract_types": dict(
                Counter(_contract_type(c.doc_id) for c in contracts).most_common(10)
            ),
        }

    return {
        "train": split_stats(train),
        "val":   split_stats(val),
        "test":  split_stats(test),
    }


def print_stats(stats: dict):
    for split, s in stats.items():
        if not s:
            continue
        log.info(
            f"\n{'─'*50}\n"
            f"  {split.upper()} ({s['n_contracts']} contracts)\n"
            f"  Clauses/doc  : {s['clauses_per_doc']['mean']} "
            f"(min {s['clauses_per_doc']['min']}, max {s['clauses_per_doc']['max']})\n"
            f"  Sentences/doc: {s['sentences_per_doc']['mean']}\n"
            f"  Boundary density: {s['boundary_density_mean']:.4f}\n"
            f"  Top contract types: "
            + ", ".join(f"{k}({v})" for k, v in list(s['contract_types'].items())[:5])
        )


# ── Main ──────────────────────────────────────────────────────────────────────


def main():
    args = parse_args()
    out  = Path(args.output_dir)

    # 1. Load
    contracts = load_cuad(
        cuad_root=args.cuad_dir,
        min_fuzzy_ratio=args.min_fuzzy_ratio,
        max_contracts=args.max_contracts,
    )

    if not contracts:
        log.error("No contracts loaded. Check --cuad_dir path.")
        return

    # 2. Split
    train, val, test = stratified_split(
        contracts, val_frac=args.val_frac, test_frac=args.test_frac, seed=args.seed
    )
    log.info(f"\nSplit: train={len(train)}, val={len(val)}, test={len(test)}")

    # 3. Write JSONL
    write_jsonl(train, out / "train.jsonl")
    write_jsonl(val,   out / "val.jsonl")
    write_jsonl(test,  out / "test.jsonl")

    # 4. Stats
    stats = compute_stats(train, val, test)
    print_stats(stats)
    with open(out / "stats.json", "w") as f:
        json.dump(stats, f, indent=2)
    log.info(f"\nStats saved → {out / 'stats.json'}")
    log.info("Data preparation complete.")

## 8. Training Loop

In [ ]:
import logging
import os
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from transformers import AutoTokenizer


logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)s  %(message)s")
log = logging.getLogger(__name__)


# ── Argument parsing ──────────────────────────────────────────────────────────


def build_optimiser(model, encoder_lr=2e-5, head_lr=1e-3, weight_decay=0.01):
    encoder_params   = list(model.sentence_encoder.parameters())
    encoder_ids      = {id(p) for p in encoder_params}
    non_encoder_params = [p for p in model.parameters() if id(p) not in encoder_ids]

    param_groups = [
        {"params": encoder_params,    "lr": encoder_lr},
        {"params": non_encoder_params,"lr": head_lr},
    ]
    return AdamW(param_groups, weight_decay=weight_decay, betas=(0.9, 0.999))


# ── Training / validation steps ───────────────────────────────────────────────

def train_epoch(
    model: HierarchicalClauseSegmenter,
    loader: DataLoader,
    optimiser: AdamW,
    device: torch.device,
    grad_accum: int,
) -> float:
    model.train()
    total_loss = 0.0
    optimiser.zero_grad()

    for step, batch in enumerate(loader):
        batch = {k: v.to(device) for k, v in batch.items()}

        loss, _ = model(
            input_ids      = batch["input_ids"],
            attention_mask = batch["attention_mask"],
            sentence_mask  = batch["sentence_mask"],
            pos_features   = batch["pos_features"],
            labels         = batch["labels"],
        )
        loss = loss / grad_accum
        loss.backward()
        total_loss += loss.item() * grad_accum

        if (step + 1) % grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimiser.step()
            optimiser.zero_grad()

    # flush remaining gradients
    if len(loader) % grad_accum != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        optimiser.zero_grad()

    return total_loss / len(loader)


@torch.no_grad()
def eval_epoch(
    model: HierarchicalClauseSegmenter,
    loader: DataLoader,
    device: torch.device,
    threshold: float,
) -> Dict[str, float]:
    model.eval()
    all_preds:  List[List[int]] = []
    all_labels: List[List[int]] = []

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        preds = model.predict(
            input_ids      = batch["input_ids"],
            attention_mask = batch["attention_mask"],
            sentence_mask  = batch["sentence_mask"],
            pos_features   = batch["pos_features"],
            threshold      = threshold,
        )
        # collect per-document sequences (ignore padding)
        masks  = batch["sentence_mask"]   # [B, N]
        labels = batch["labels"]          # [B, N]
        for i in range(preds.shape[0]):
            length = masks[i].sum().item()
            all_preds.append(preds[i, :length].cpu().tolist())
            all_labels.append(labels[i, :length].cpu().tolist())

    return compute_metrics(all_preds, all_labels)


# ── Main training loop ────────────────────────────────────────────────────────

def train(args):
    torch.manual_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log.info(f"Using device: {device}")

    # ── Tokenizer & datasets ──────────────────────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained(args.encoder)

    log.info("Loading training data …")
    train_ds = ContractDataset.from_jsonl(
        args.train, tokenizer, max_sent_len=args.max_sent_len
    )
    log.info("Loading validation data …")
    val_ds = ContractDataset.from_jsonl(
        args.val, tokenizer, max_sent_len=args.max_sent_len
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=args.batch_size,
        shuffle=True,
        collate_fn=collate_contracts,
        num_workers=2,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=args.batch_size,
        shuffle=False,
        collate_fn=collate_contracts,
        num_workers=2,
    )

    # ── Model ─────────────────────────────────────────────────────────────────
    cfg = SegmenterConfig(
        encoder_model_name=args.encoder,
        freeze_encoder_epochs=args.freeze_epochs,
        use_crf=args.use_crf,
        use_focal_loss=args.focal_loss,
    )
    model = HierarchicalClauseSegmenter(cfg).to(device)

    if args.freeze_epochs > 0:
        log.info(f"Freezing encoder for first {args.freeze_epochs} epoch(s)")
        model.freeze_encoder()

    optimiser = build_optimiser(model, args)
    scheduler = CosineAnnealingLR(optimiser, T_max=args.epochs, eta_min=1e-6)

    # ── Training loop ─────────────────────────────────────────────────────────
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    best_f1      = 0.0
    patience_cnt = 0

    for epoch in range(1, args.epochs + 1):

        # unfreeze encoder after freeze_epochs
        if epoch == args.freeze_epochs + 1:
            log.info("Unfreezing encoder")
            model.unfreeze_encoder()

        train_loss = train_epoch(model, train_loader, optimiser, device, args.grad_accum)
        val_metrics = eval_epoch(model, val_loader, device, args.threshold)
        scheduler.step()

        f1 = val_metrics["boundary_f1"]
        log.info(
            f"Epoch {epoch:02d}  train_loss={train_loss:.4f}  "
            f"val_P={val_metrics['precision']:.4f}  "
            f"val_R={val_metrics['recall']:.4f}  "
            f"val_F1={f1:.4f}  "
            f"WindowDiff={val_metrics.get('window_diff', -1):.4f}"
        )

        if f1 > best_f1:
            best_f1 = f1
            patience_cnt = 0
            _save_checkpoint(model, tokenizer, cfg, output_dir / "best_model")
            log.info(f"  ✓ New best F1={best_f1:.4f} — checkpoint saved")
        else:
            patience_cnt += 1
            log.info(f"  No improvement ({patience_cnt}/{args.patience})")
            if patience_cnt >= args.patience:
                log.info("Early stopping.")
                break

    log.info(f"Training complete. Best val F1: {best_f1:.4f}")


def _save_checkpoint(model, tokenizer, cfg, path: Path):
    path.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), path / "model.pt")
    tokenizer.save_pretrained(str(path))
    import json, dataclasses
    with open(path / "config.json", "w") as f:
        json.dump(dataclasses.asdict(cfg), f, indent=2)

## 9. End-to-End Run

> **Set `CUAD_DIR` below before running.**

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
CUAD_DIR      = "./CUAD_v1/CUAD_v1"   # <-- SET THIS to your CUAD_v1 folder
OUTPUT_DIR    = "./runs/exp1"
ENCODER       = "nlpaueb/legal-bert-base-uncased"
EPOCHS        = 6
BATCH_SIZE    = 4
GRAD_ACCUM    = 4
ENCODER_LR    = 2e-5
HEAD_LR       = 1e-3
FREEZE_EPOCHS = 2
PATIENCE      = 3
MAX_SENT_LEN  = 128
MAX_CONTRACTS = None  # set e.g. 20 for a quick smoke-test on CPU
# ─────────────────────────────────────────────────────────────────────────────

data_dir   = Path(OUTPUT_DIR) / "data"
train_path = data_dir / "train.jsonl"
val_path   = data_dir / "val.jsonl"
test_path  = data_dir / "test.jsonl"

# Step 1 — prepare data (skipped if files already exist)
if train_path.exists():
    print("JSONL files found — skipping data prep.")
else:
    print("Preparing CUAD data …")
    contracts = load_cuad(CUAD_DIR, max_contracts=MAX_CONTRACTS)
    train_contracts, val_contracts, test_contracts = stratified_split(contracts)
    write_jsonl(train_contracts, train_path)
    write_jsonl(val_contracts,   val_path)
    write_jsonl(test_contracts,  test_path)

# Step 2 — train
print("\nLoading tokenizer …")
tokenizer = AutoTokenizer.from_pretrained(ENCODER)

print("Loading datasets …")
train_ds = ContractDataset.from_jsonl(train_path, tokenizer, MAX_SENT_LEN)
val_ds   = ContractDataset.from_jsonl(val_path,   tokenizer, MAX_SENT_LEN)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  collate_fn=collate_contracts, num_workers=2)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, collate_fn=collate_contracts, num_workers=2)

cfg   = SegmenterConfig(encoder_model_name=ENCODER, freeze_encoder_epochs=FREEZE_EPOCHS, use_crf=True)
model = HierarchicalClauseSegmenter(cfg).to(DEVICE)

if FREEZE_EPOCHS > 0:
    model.freeze_encoder()
    print(f"Encoder frozen for first {FREEZE_EPOCHS} epoch(s)")

optimiser = build_optimiser(model, encoder_lr=ENCODER_LR, head_lr=HEAD_LR)
scheduler = CosineAnnealingLR(optimiser, T_max=EPOCHS, eta_min=1e-6)

best_f1, patience_cnt, history = 0.0, 0, []

for epoch in range(1, EPOCHS + 1):
    if epoch == FREEZE_EPOCHS + 1:
        model.unfreeze_encoder()
        print("Encoder unfrozen")

    train_loss  = train_epoch(model, train_loader, optimiser, DEVICE, GRAD_ACCUM)
    val_metrics = eval_epoch(model, val_loader, DEVICE, threshold=0.5)
    scheduler.step()

    f1 = val_metrics["boundary_f1"]
    history.append({"epoch": epoch, "loss": train_loss, **val_metrics})
    print(f"Epoch {epoch:02d} | loss={train_loss:.4f} | "
          f"P={val_metrics['precision']:.4f} R={val_metrics['recall']:.4f} "
          f"F1={f1:.4f} | WD={val_metrics['window_diff']:.4f}")

    ckpt_dir = Path(OUTPUT_DIR) / "best_model"
    if f1 > best_f1:
        best_f1, patience_cnt = f1, 0
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), ckpt_dir / "model.pt")
        tokenizer.save_pretrained(str(ckpt_dir))
        with open(ckpt_dir / "config.json", "w") as fh:
            json.dump(dataclasses.asdict(cfg), fh, indent=2)
        print(f"  ✓ Checkpoint saved (F1={best_f1:.4f})")
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print("Early stopping."); break

print(f"\nTraining done. Best val F1: {best_f1:.4f}")

# Step 3 — sweep threshold on validation set
print("\nSweeping threshold …")
all_probs, all_val_labels = [], []
model.eval()
with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        _, logits = model(batch["input_ids"], batch["attention_mask"],
                          batch["sentence_mask"], batch["pos_features"])
        probs = torch.softmax(logits, dim=-1)[:, :, 1]
        for i in range(probs.shape[0]):
            n = batch["sentence_mask"][i].sum().item()
            all_probs.append(probs[i, :n].cpu().tolist())
            all_val_labels.append(batch["labels"][i, :n].cpu().tolist())

BEST_THRESHOLD, best_t_metrics = find_best_threshold(all_probs, all_val_labels)
print(f"Best threshold: {BEST_THRESHOLD:.2f}  "
      f"F1={best_t_metrics['boundary_f1']:.4f}  WD={best_t_metrics['window_diff']:.4f}")

# Step 4 — evaluate on test set
print("\nEvaluating on test set …")
test_ds     = ContractDataset.from_jsonl(test_path, tokenizer, MAX_SENT_LEN)
test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, collate_fn=collate_contracts)
test_metrics = eval_epoch(model, test_loader, DEVICE, threshold=BEST_THRESHOLD)

print("\n── Test Results ────────────────────────")
for k, v in test_metrics.items():
    print(f"  {k:20s}: {v:.4f}")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
with open(Path(OUTPUT_DIR) / "test_results.json", "w") as fh:
    json.dump(test_metrics, fh, indent=2)
with open(Path(OUTPUT_DIR) / "best_threshold.json", "w") as fh:
    json.dump({"threshold": BEST_THRESHOLD, "metrics": best_t_metrics}, fh, indent=2)
print("Results saved.")

## 10. Inference Demo

In [ ]:
import argparse
import dataclasses
import json
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional

import torch
from transformers import AutoTokenizer



# ── Clause span output ────────────────────────────────────────────────────────

@dataclass
class ClauseSpan:
    index:      int
    text:       str
    start_char: int
    end_char:   int


# ── Segmenter class ───────────────────────────────────────────────────────────

class ClauseSegmenter:

    def __init__(
        self,
        model: HierarchicalClauseSegmenter,
        tokenizer,
        device: torch.device,
        threshold: float = 0.5,
        max_sent_len: int = 128,
        min_clause_sentences: int = 1,
    ):
        self.model     = model
        self.tokenizer = tokenizer
        self.device    = device
        self.threshold = threshold
        self.max_sent_len = max_sent_len
        self.min_clause_sentences = min_clause_sentences

        self.model.eval()
        self.model.to(device)

    # ── Public API ────────────────────────────────────────────────────────────

    def segment(self, text: str) -> List[ClauseSpan]:
        """
        Segment `text` into clauses.
        Returns a list of ClauseSpan objects in document order.
        """
        sentences = split_contract(text)
        if not sentences:
            return []

        # Prepare tokenised document (no labels needed)
        raw = RawContract(doc_id="inference", text=text, clauses=[])
        tok = tokenise_contract(raw, self.tokenizer, self.max_sent_len)

        batch = collate_contracts([tok])
        batch = {k: v.to(self.device) for k, v in batch.items()}

        preds = self.model.predict(
            input_ids      = batch["input_ids"],
            attention_mask = batch["attention_mask"],
            sentence_mask  = batch["sentence_mask"],
            pos_features   = batch["pos_features"],
            threshold      = self.threshold,
        )

        labels = preds[0, : len(sentences)].cpu().tolist()   # [N]

        # Postprocess: merge tiny clauses
        labels = self._postprocess(labels, sentences)

        return self._build_spans(text, sentences, labels)

    def segment_batch(self, texts: List[str]) -> List[List[ClauseSpan]]:
        """Segment multiple contracts in one call (more efficient)."""
        return [self.segment(t) for t in texts]

    # ── Postprocessing ────────────────────────────────────────────────────────

    def _postprocess(self, labels: List[int], sentences) -> List[int]:
        """
        Merge clauses that are shorter than min_clause_sentences into
        their predecessor.
        """
        if self.min_clause_sentences <= 1:
            return labels

        # Identify clause start indices
        starts = [i for i, l in enumerate(labels) if l == 1]
        if not starts:
            return labels

        # Compute clause lengths
        starts_ext = starts + [len(labels)]
        clause_lens = [starts_ext[i + 1] - starts_ext[i] for i in range(len(starts))]

        # Suppress short clause starts (merge into previous)
        merged = labels[:]
        for i, (start_idx, clen) in enumerate(zip(starts, clause_lens)):
            if clen < self.min_clause_sentences and i > 0:
                merged[start_idx] = 0   # absorb into previous

        return merged

    # ── Span construction ─────────────────────────────────────────────────────

    @staticmethod
    def _build_spans(text: str, sentences, labels: List[int]) -> List[ClauseSpan]:
        spans: List[ClauseSpan] = []
        clause_idx = 0
        current_start = None
        current_sentences: List[str] = []

        def flush(end_char: int):
            nonlocal clause_idx, current_start, current_sentences
            if current_start is None or not current_sentences:
                return
            clause_text = " ".join(current_sentences)
            spans.append(ClauseSpan(
                index=clause_idx,
                text=clause_text,
                start_char=current_start,
                end_char=end_char,
            ))
            clause_idx += 1
            current_start = None
            current_sentences = []

        for sent, label in zip(sentences, labels):
            if label == 1:
                if current_start is not None:
                    flush(sent.start_char)
                current_start = sent.start_char
                current_sentences = [sent.text]
            else:
                if current_start is None:
                    # document doesn't start with a B label — treat first as B
                    current_start = sent.start_char
                current_sentences.append(sent.text)

        if current_start is not None:
            flush(len(text))

        return spans

    # ── Serialisation ─────────────────────────────────────────────────────────

    @classmethod
    def from_pretrained(
        cls,
        checkpoint_dir: str | Path,
        device: Optional[torch.device] = None,
        threshold: float = 0.5,
        max_sent_len: int = 128,
    ) -> "ClauseSegmenter":
        checkpoint_dir = Path(checkpoint_dir)
        if device is None:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Load config
        with open(checkpoint_dir / "config.json") as f:
            cfg_dict = json.load(f)
        cfg = SegmenterConfig(**cfg_dict)

        # Load model weights
        model = HierarchicalClauseSegmenter(cfg)
        state = torch.load(checkpoint_dir / "model.pt", map_location=device)
        model.load_state_dict(state)
        model.eval()

        tokenizer = AutoTokenizer.from_pretrained(str(checkpoint_dir))

        return cls(
            model=model,
            tokenizer=tokenizer,
            device=device,
            threshold=threshold,
            max_sent_len=max_sent_len,
        )



In [ ]:
# ── Demo ─────────────────────────────────────────────────────────────────────
# Point this at any contract .txt, or leave None to use the built-in example.
DEMO_CONTRACT_PATH = None  # e.g. "./CUAD_v1/CUAD_v1/full_contract_txt/AMAZON_ServiceAgreement.txt"

SAMPLE_CONTRACT = """
NON-DISCLOSURE AGREEMENT

This Non-Disclosure Agreement ("Agreement") is entered into as of January 1, 2024,
by and between Acme Corp ("Disclosing Party") and Beta LLC ("Receiving Party").

1. CONFIDENTIAL INFORMATION
"Confidential Information" means any information disclosed by the Disclosing Party
that is designated as confidential or that reasonably should be understood to be
confidential given the nature of the information and the circumstances of disclosure.

2. OBLIGATIONS OF RECEIVING PARTY
The Receiving Party agrees to hold the Confidential Information in strict confidence
and not to disclose it to any third parties without the prior written consent of the
Disclosing Party.

3. TERM
This Agreement shall remain in effect for a period of two (2) years from the date
of execution, unless earlier terminated by mutual written agreement of the parties.

4. GOVERNING LAW
This Agreement shall be governed by and construed in accordance with the laws of
the State of Delaware, without regard to its conflict of law provisions.

5. ENTIRE AGREEMENT
This Agreement constitutes the entire agreement between the parties with respect
to the subject matter hereof and supersedes all prior agreements and understandings.
"""

ckpt_path = Path(OUTPUT_DIR) / "best_model"

if not ckpt_path.exists():
    print(f"No checkpoint at {ckpt_path}. Run Section 9 first.")
else:
    seg = ClauseSegmenter.from_pretrained(
        ckpt_path,
        device=DEVICE,
        threshold=BEST_THRESHOLD if 'BEST_THRESHOLD' in dir() else 0.45,
    )

    if DEMO_CONTRACT_PATH and Path(DEMO_CONTRACT_PATH).exists():
        text = Path(DEMO_CONTRACT_PATH).read_text(encoding="utf-8", errors="replace")
    else:
        text = SAMPLE_CONTRACT

    clauses = seg.segment(text)
    print(f"Found {len(clauses)} clauses\n{'─'*50}")
    for c in clauses:
        print(f"[{c.index:02d}] chars {c.start_char}–{c.end_char}")
        print(f"     {c.text[:120]}...")
        print()